# H/k Sweep Analysis

This notebook is the compact analysis surface for the current H/k prototype. It covers the supported orchestrator tasks:

- GitLab Task 44: short reference task for continuation planning with different `H` values.
- Shopping Task 118: longer product-navigation task where one planner subgoal can require multiple concrete browser actions.

Interpretation:

- `H` controls how many subgoals the planner exposes in one planner call. `H=0` means full plan.
- `k` controls how often the runtime evaluator validates after concrete executor/browser actions.
- WebArena-Verified remains the official final evaluator via `eval-tasks`.

In [ ]:
from pathlib import Path
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

OFFICIAL_REPO = ROOT / 'external' / 'webarena-verified'
SWEEP_BASE = OFFICIAL_REPO / 'output' / 'hk-sweep'

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_columns', 80)

ROOT, SWEEP_BASE

def read_json(path: Path):
    return json.loads(path.read_text())



## Optional: Run All Current H/k Sweeps

Run this cell when you want to regenerate the current local comparison. It can take several minutes because every row calls Ollama, BrowserGym, and the official WebArena-Verified evaluator.

The output roots are split by site so GitLab and Shopping results do not overwrite each other.

In [ ]:
RUN_SWEEPS = False
MODEL = 'gemma4:26b'

SWEEP_COMMANDS = [
    [
        'uv', 'run', 'python', 'scripts/run_hk_sweep.py',
        '--site', 'gitlab',
        '--output-root', 'output/hk-sweep/gitlab',
        '--hs', '0', '1', '2',
        '--ks', '1', '2',
        '--model', MODEL,
    ],
    [
        'uv', 'run', 'python', 'scripts/run_hk_sweep.py',
        '--site', 'shopping',
        '--output-root', 'output/hk-sweep/shopping',
        '--hs', '0',
        '--ks', '1', '2',
        '--model', MODEL,
    ],
]

if RUN_SWEEPS:
    for cmd in SWEEP_COMMANDS:
        print('$', ' '.join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print('Sweep execution skipped. Set RUN_SWEEPS = True to regenerate results.')

## Optional: Run Supporting Examples

These commands regenerate the non-H/k examples used as current coverage checks: service reachability, official hardcoded tasks including Reddit, and planner previews for every enabled local site except Map.

In [ ]:
RUN_SUPPORTING_EXAMPLES = False

SUPPORTING_COMMANDS = [
    ['uv', 'run', 'python', 'scripts/run_services_probe.py', '--sites', 'shopping', 'shopping_admin', 'reddit', 'gitlab', 'wikipedia'],
    ['uv', 'run', 'python', 'scripts/run_hardcoded_tasks.py', '--sites', 'shopping', 'shopping_admin', 'reddit', 'gitlab'],
    ['uv', 'run', 'python', 'scripts/preview_planner.py', '--planner-mode', 'ollama', '--model', MODEL, '--sites', 'shopping', 'shopping_admin', 'reddit', 'gitlab', 'wikipedia', '--h', '0'],
]

if RUN_SUPPORTING_EXAMPLES:
    for cmd in SUPPORTING_COMMANDS:
        print('$', ' '.join(cmd))
        subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print('Supporting examples skipped. Set RUN_SUPPORTING_EXAMPLES = True to regenerate them.')

## Service Probe Coverage

The service probe verifies that every included local site can be opened through BrowserGym. It is not a task-solving run.

In [ ]:
SERVICE_SUMMARY = OFFICIAL_REPO / 'output' / 'service-probe' / 'summary.json'

if SERVICE_SUMMARY.exists():
    service_summary = read_json(SERVICE_SUMMARY)
    service_df = pd.DataFrame(service_summary.get('results', []))
    display(service_df[[
        'site', 'status', 'task_id', 'task_type', 'start_url', 'final_url', 'page_title', 'error'
    ]])
    print('excluded:', service_summary.get('excluded_sites', {}))
else:
    print('No service-probe summary found yet:', SERVICE_SUMMARY)

## Planner Preview Coverage

Planner previews call the LLM planner and write plan artifacts, but they do not execute browser actions.

In [ ]:
PLANNER_PREVIEW_SUMMARY = OFFICIAL_REPO / 'output' / 'planner-preview' / 'summary.json'

if PLANNER_PREVIEW_SUMMARY.exists():
    planner_preview = read_json(PLANNER_PREVIEW_SUMMARY)
    preview_df = pd.DataFrame(planner_preview.get('results', []))
    if preview_df.empty:
        print('Planner preview summary exists but has no rows.')
    else:
        display(preview_df[[
            'site', 'task_id', 'planner_mode', 'model_name', 'h', 'subgoal_count', 'warnings', 'output_dir'
        ]])
else:
    print('No planner-preview summary found yet:', PLANNER_PREVIEW_SUMMARY)

## Load Sweep Summaries

In [ ]:
summary_paths = sorted((SWEEP_BASE).glob('*/summary.json'))
print('summary files:')
for path in summary_paths:
    print('-', path.relative_to(ROOT))

rows = []
for path in summary_paths:
    summary = read_json(path)
    for row in summary.get('rows', []):
        row = dict(row)
        row['summary_file'] = str(path)
        rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print('No sweep rows found. Run the optional sweep cell first.')
else:
    df['runtime_s'] = df['total_runtime_ms'] / 1000
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    df = df.sort_values(['site', 'task_id', 'h', 'k']).reset_index(drop=True)
    display(df[[
        'site', 'task_id', 'h', 'k', 'score', 'success', 'total_steps',
        'num_planner_calls', 'num_plan_subgoals_generated', 'total_tokens',
        'prompt_tokens', 'completion_tokens', 'runtime_s', 'output_dir'
    ]])

## Score And Success Overview

In [ ]:
if not df.empty:
    overview = df.groupby(['site', 'task_id'], as_index=False).agg(
        runs=('score', 'size'),
        successful_runs=('success', 'sum'),
        min_score=('score', 'min'),
        max_score=('score', 'max'),
        mean_runtime_s=('runtime_s', 'mean'),
        mean_tokens=('total_tokens', 'mean'),
    )
    display(overview)

In [ ]:
if not df.empty:
    pivot = df.pivot_table(index=['site', 'task_id', 'h'], columns='k', values='score', aggfunc='first')
    display(pivot)

## Runtime, Tokens, And Planner Calls

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for site, site_df in df.groupby('site'):
        label = site
        x = site_df['h'].astype(str) + '/k' + site_df['k'].astype(str)
        axes[0].plot(x, site_df['runtime_s'], marker='o', label=label)
        axes[1].plot(x, site_df['total_tokens'], marker='o', label=label)
        axes[2].plot(x, site_df['num_planner_calls'], marker='o', label=label)
    axes[0].set_title('Runtime seconds')
    axes[1].set_title('Total tokens')
    axes[2].set_title('Planner calls')
    for ax in axes:
        ax.set_xlabel('H/k')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()

## Trace Inspection Helpers

The official score tells whether the final answer passed. The trace files show why `H` and `k` matter internally.

In [ ]:
def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return pd.DataFrame(rows)

def artifact_paths(output_dir: str) -> dict[str, Path]:
    base = Path(output_dir)
    return {
        'plan': base / 'plan.json',
        'step_trace': base / 'step_trace.jsonl',
        'evaluator_signals': base / 'evaluator_signals.jsonl',
        'controller_decisions': base / 'controller_decisions.jsonl',
        'run_summary': base / 'run_summary.json',
        'eval_result': base / 'eval_result.json',
    }

def show_run(site: str, h: int, k: int):
    match = df[(df['site'] == site) & (df['h'] == h) & (df['k'] == k)]
    if match.empty:
        raise ValueError(f'No run found for site={site}, h={h}, k={k}')
    row = match.iloc[0]
    paths = artifact_paths(row['output_dir'])
    print('output_dir:', row['output_dir'])
    print('score:', row['score'], 'success:', row['success'], 'final_url:', row['final_url'])
    display(read_json(paths['run_summary']) if paths['run_summary'].exists() else {})
    return paths

## GitLab Example: H Controls Continuation Planning

Task 44 is short. `k` has limited visible effect because there are only two concrete actions, but `H=1` shows continuation planning because the planner emits one subgoal per call.

In [ ]:
if not df.empty and 'gitlab' in set(df['site']):
    gitlab_rows = df[df['site'] == 'gitlab'][[
        'site', 'task_id', 'h', 'k', 'score', 'total_steps',
        'num_planner_calls', 'num_plan_subgoals_generated', 'total_tokens', 'runtime_s'
    ]]
    display(gitlab_rows)

## Shopping Example: k Controls Action-Level Validation

Shopping Task 118 is the first current example where one high-level planner subgoal can require multiple concrete browser actions. With `k=1`, the runtime evaluator reports the search page as partial progress and the product page as completion. With `k=2`, it only evaluates after the second action.

In [ ]:
if not df.empty and 'shopping' in set(df['site']):
    shopping_rows = df[df['site'] == 'shopping'][[
        'site', 'task_id', 'h', 'k', 'score', 'total_steps',
        'num_planner_calls', 'num_plan_subgoals_generated', 'total_tokens', 'runtime_s', 'output_dir'
    ]]
    display(shopping_rows)

In [ ]:
if not df.empty and 'shopping' in set(df['site']):
    for k_value in sorted(df[df['site'] == 'shopping']['k'].unique()):
        paths = show_run('shopping', h=0, k=int(k_value))
        print('\nstep_trace')
        display(read_jsonl(paths['step_trace'])[['step_index', 'subgoal_id', 'action', 'url_before', 'url_after', 'status']])
        print('\nevaluator_signals')
        signals = read_jsonl(paths['evaluator_signals'])
        display(signals[['step_index', 'subgoal_id', 'progress_score', 'subgoal_done', 'reason', 'recommended_intervention']])

## Other Current Official Examples

The H/k orchestrator currently covers GitLab and Shopping. The wider local benchmark sanity set is still visible through the hardcoded official tasks, including Reddit Task 27. These rows are useful as a baseline before adding more site-specific H/k executors.

In [ ]:
HARDCODED_SUMMARY = OFFICIAL_REPO / 'output' / 'hardcoded-tasks' / 'summary.json'

if HARDCODED_SUMMARY.exists():
    hardcoded = read_json(HARDCODED_SUMMARY).get('results', [])
    hardcoded_df = pd.DataFrame(hardcoded)
    display(hardcoded_df[[
        'site', 'task_id', 'task_type', 'status', 'success', 'official_score',
        'total_runtime_ms', 'final_url', 'output_dir'
    ]])
else:
    print('No hardcoded summary found yet:', HARDCODED_SUMMARY)

### Commands For Non-H/k Examples

These commands exercise the currently available non-H/k examples and write official evaluator artifacts where a real WebArena-Verified task id exists.

In [ ]:
NON_HK_COMMANDS = [
    ['uv', 'run', 'python', 'scripts/run_services_probe.py', '--sites', 'shopping', 'shopping_admin', 'reddit', 'gitlab', 'wikipedia'],
    ['uv', 'run', 'python', 'scripts/run_hardcoded_tasks.py', '--sites', 'shopping', 'shopping_admin', 'reddit', 'gitlab'],
    ['uv', 'run', 'python', 'scripts/preview_planner.py', '--planner-mode', 'ollama', '--model', MODEL, '--sites', 'shopping', 'shopping_admin', 'reddit', 'gitlab', 'wikipedia', '--h', '0'],
]

for command in NON_HK_COMMANDS:
    print(' '.join(command))

## Interpretation Snapshot

Use this cell as a lightweight sanity check before writing thesis notes from the run data.

In [ ]:
if not df.empty:
    all_passed = bool((df['score'] == 1.0).all() and (df['returncode'] == 0).all())
    print('all current H/k sweep rows passed official eval:', all_passed)
    print('sites covered:', ', '.join(sorted(df['site'].unique())))
    print('runs:', len(df))

    if 'shopping' in set(df['site']):
        signal_counts = []
        for _, row in df[df['site'] == 'shopping'].iterrows():
            signals = read_jsonl(artifact_paths(row['output_dir'])['evaluator_signals'])
            signal_counts.append({'k': row['k'], 'evaluator_signal_count': len(signals)})
        display(pd.DataFrame(signal_counts).sort_values('k'))